# MR²-AttUNet Tutorial

Companion notebook to the paper *Bridging the Gap: Supervised MRI Motion Artifact Correction via Data Synthesis: A CBAM-U-Net Approach*. The model is branded **MR²-AttUNet** (Motion Removal × Multi-Resolution Attention U-Net) in this code release; the Python package is imported as `cbam_unet`.

This notebook provides a slim, end-to-end smoke test of the pipeline:

1. Synthesise a tiny paired dataset from a folder of MR-ART originals.
2. Train MR²-AttUNet for a few epochs to verify the pipeline.
3. Run inference on a corrupted scan and evaluate metrics.

For full training, use `scripts/train.py` instead. See `README.md` for the full quickstart.


## 0. Setup

Install the package in editable mode (run this once from the repo root):

```bash
pip install -e .
```


In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / 'src'))

import numpy as np
import tensorflow as tf

from cbam_unet import artifact_removal_model
from cbam_unet.data import make_paired_dataset, count_examples
from cbam_unet.synth import generate_split
from cbam_unet.metrics import psnr, ssim

## 1. Synthesise paired data

Point `MRART_ORIGINALS` at a folder of motion-free MR-ART PNGs. The cell below writes
paired `Original/` and `Corrupted/` PNGs under `data/train/` and `data/val/` using the
stochastic k-space perturbation framework described in the paper.

For a quick smoke test, point at a folder with ~20 images.


In [ ]:
MRART_TRAIN_ORIGINALS = REPO_ROOT / 'path' / 'to' / 'MRART_training_data' / 'Original'
MRART_VAL_ORIGINALS   = REPO_ROOT / 'path' / 'to' / 'MRART_val_data' / 'Original'
DATA_ROOT = REPO_ROOT / 'data'

ARTIFACT_PARAMS = dict(
    motion_type='both', shift_x=2, shift_y=0, angle=3,
    number_of_lines=50, width=3, region='random',
)

np.random.seed(0)
if MRART_TRAIN_ORIGINALS.is_dir():
    generate_split(MRART_TRAIN_ORIGINALS, DATA_ROOT / 'train', ARTIFACT_PARAMS, 'TRAIN')
if MRART_VAL_ORIGINALS.is_dir():
    generate_split(MRART_VAL_ORIGINALS, DATA_ROOT / 'val', ARTIFACT_PARAMS, 'VAL')

## 2. Build and train

We build MR²-AttUNet at 256x256 and train for a small number of epochs as a smoke test.


In [ ]:
tf.keras.utils.set_random_seed(0)

TRAIN_DIR = DATA_ROOT / 'train'
VAL_DIR   = DATA_ROOT / 'val'

print('Train images:', count_examples(TRAIN_DIR))
print('Val   images:', count_examples(VAL_DIR))

train_ds = make_paired_dataset(TRAIN_DIR, image_size=256, batch_size=4)
val_ds   = make_paired_dataset(VAL_DIR,   image_size=256, batch_size=4)

model = artifact_removal_model(image_resolution=256)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='mse',
    metrics=['accuracy'],
)
model.summary()

history = model.fit(train_ds, validation_data=val_ds, epochs=2)

## 3. Inference and metrics on a single example


In [ ]:
import cv2
import matplotlib.pyplot as plt

corr_files = sorted((VAL_DIR / 'Corrupted').glob('*.png'))
orig_files = sorted((VAL_DIR / 'Original').glob('*.png'))

if corr_files and orig_files:
    deg = cv2.imread(str(corr_files[0]), cv2.IMREAD_GRAYSCALE)
    ref = cv2.imread(str(orig_files[0]), cv2.IMREAD_GRAYSCALE)
    deg = cv2.resize(deg, (256, 256)).astype(np.float32) / 255.0
    ref = cv2.resize(ref, (256, 256)).astype(np.float32) / 255.0

    pred = model.predict(deg[np.newaxis, :, :, np.newaxis], verbose=0)
    print(f'PSNR = {psnr(ref[np.newaxis, :, :, np.newaxis], pred):.2f} dB')
    print(f'SSIM = {ssim(ref[np.newaxis, :, :, np.newaxis], pred):.4f}')

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(deg, cmap='gray');                axes[0].set_title('Input (Corrupted)'); axes[0].axis('off')
    axes[1].imshow(pred[0, :, :, 0], cmap='gray');   axes[1].set_title('Model Output');       axes[1].axis('off')
    axes[2].imshow(ref, cmap='gray');                axes[2].set_title('Ground Truth');       axes[2].axis('off')
    plt.tight_layout(); plt.show()
else:
    print('No paired files in', VAL_DIR)